# 11.1 — RL Framing

Reinforcement learning (RL) frames learning as an agent repeatedly acting in an environment, receiving rewards, and using delayed consequences to improve future choices. In this lesson, we build the bookkeeping from scratch: states, actions, transitions, rewards, discounted returns, one-step bootstrap targets, policies, and exploration pressure.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build RL framing one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the agent-environment loop is not a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random choices, and vectorized expectation arithmetic.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for simulated episodes.

### 1. The agent-environment loop

An RL problem starts with a loop, not a static table. The **agent** observes a state, chooses an action, the **environment** moves to a next state, and the agent receives a reward. Here the world is a tiny corridor: state `0` is start, state `1` is a middle square, and state `2` is a goal. Action `0` moves left and action `1` moves right. The goal gives reward `1`; other transitions give `0`.

In [ ]:
states_w = np.array([0, 1, 2])  # three corridor states.
actions_w = np.array([0, 1])    # 0 = left, 1 = right.
T_w = np.array([[0, 1],          # from state 0: left stays 0, right goes 1.
                [0, 2],          # from state 1: left goes 0, right goes 2.
                [2, 2]])         # goal is terminal-like: both actions stay at 2.
R_w = np.array([[0., 0.],        # rewards indexed by current state and action.
                [0., 1.],
                [0., 0.]])
print("transition table T[s,a]:\n", T_w)
print("reward table R[s,a]:\n", R_w)

▶ What you'll see: the action table shows how a choice changes the next state, while the reward table shows that only moving right from state 1 pays.

In [ ]:
s_w = 0
trajectory_w = []
for t_w in range(3):
    a_w = 1  # a simple policy: always move right.
    r_w = R_w[s_w, a_w]
    s_next_w = T_w[s_w, a_w]
    trajectory_w.append((s_w, a_w, r_w, s_next_w))
    s_w = s_next_w
print("(state, action, reward, next_state):")
for row_w in trajectory_w:
    print(row_w)
assert trajectory_w[1] == (1, 1, 1.0, 2)

▶ What you'll see: the reward arrives on the second transition, after the first action merely positioned the agent.

In [ ]:
plt.figure(figsize=(5, 1.8))
plt.scatter(states_w, np.zeros_like(states_w), s=[180, 180, 220], color=["gray", "orange", "green"])
for i_w, label_w in enumerate(["start", "middle", "goal"]):
    plt.text(i_w, 0.05, label_w, ha="center")
plt.plot([0, 1, 2], [0, 0, 0], color="black", linewidth=1)
plt.yticks([]); plt.xticks(states_w); plt.title("1: tiny environment state line")
plt.show()

▶ What you'll see: a three-state line where the first move creates access to the later reward.

*Why it's done this way:* RL needs explicit transition bookkeeping because an action is not just a label to predict; it changes the next input distribution. The tuple `(s, a, r, s')` is the minimal record that says what the agent controlled, what the environment returned, and what future state the action made possible.

### 2. Reward is immediate; return is discounted consequence

A reward is one time step's signal. A **return** adds future rewards with discount powers: $G_t=r_t+\gamma r_{t+1}+\gamma^2r_{t+2}+\cdots$. Discounting keeps delayed payoffs meaningful while making nearer evidence count more. With rewards `[1, 0, 2]` and $\gamma=0.9$, the lesson's return is $2.620$.

In [ ]:
rewards_w = np.array([1., 0., 2.])
gamma_w = 0.9
powers_w = gamma_w ** np.arange(len(rewards_w))
terms_w = powers_w * rewards_w
print("discount powers:", np.round(powers_w, 3))
print("discounted terms:", np.round(terms_w, 3))

▶ What you'll see: the third reward is multiplied by `0.9² = 0.81`, so it contributes `1.62` instead of `2`.

In [ ]:
G_w = float(np.sum(terms_w))
print("discounted return G:", round(G_w, 3))
assert round(G_w, 3) == 2.620

▶ What you'll see: `1 + 0 + 1.620 = 2.620`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["r0", "γ r1", "γ² r2"], terms_w, color="teal")
plt.axhline(0, color="black", linewidth=0.7)
plt.title("2: pieces of a discounted return")
plt.ylabel("contribution to G")
plt.show()

▶ What you'll see: the delayed payoff still dominates because it is large, but discounting visibly shrinks it.

*Why it's done this way:* optimizing immediate reward would ignore the action that sets up a future payoff. The discount factor is the mathematical compromise: it lets the agent value consequences while keeping infinite-horizon sums finite and making distant estimates less dominant than nearby evidence.

### 3. Values and the one-step bootstrap target

A value is a prediction of return. A state value $V(s)$ predicts consequence from a state; an action value $Q(s,a)$ predicts consequence after choosing an action. A one-step target uses the observed reward plus the current estimate of the next state: $y=r+\gamma V(s')$. Then a learning rate $\alpha$ moves the old estimate partway toward that target.

In [ ]:
V_w = np.array([0.3, 0.8, 0.0])  # current estimates for states 0, 1, 2.
r_obs_w = 1.0
s_next_w = 1
bootstrap_target_w = r_obs_w + gamma_w * V_w[s_next_w]
print("V table:", V_w)
print("target y = r + γ V(s'):", round(bootstrap_target_w, 3))
assert round(bootstrap_target_w, 3) == 1.720

▶ What you'll see: the observed reward `1` is augmented by discounted next-state value `0.720`.

In [ ]:
q_old_w = 0.4
alpha_w = 0.5
q_new_w = q_old_w + alpha_w * (bootstrap_target_w - q_old_w)
print("old Q:", q_old_w, "new Q:", round(q_new_w, 3))
assert round(q_new_w, 3) == 1.060

▶ What you'll see: the estimate moves halfway from `0.4` toward the target `1.72`, landing at `1.06`.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["old Q", "target y", "new Q"], [q_old_w, bootstrap_target_w, q_new_w], color=["gray", "black", "seagreen"])
plt.title("3: a bootstrap update is a partial move")
plt.ylabel("value")
plt.show()

▶ What you'll see: the new estimate sits between the old estimate and the one-step target.

*Why it's done this way:* waiting for full returns can be high-variance and slow, while fully replacing an estimate with one noisy target can be unstable. The bootstrap target trades variance for bias, and the learning rate controls how much one transition is allowed to rewrite accumulated knowledge.

### 4. A policy turns scores into action probabilities

A policy $\pi(a\mid s)$ says how likely each action is in a state. One common parameterization stores action scores (logits) and applies softmax: $\pi_i=e^{z_i}/\sum_j e^{z_j}$. This turns arbitrary real scores into probabilities that sum to 1, so expected reward is a probability-weighted average.

In [ ]:
logits_w = np.array([1., 0.])
exp_w = np.exp(logits_w - np.max(logits_w))  # stable softmax: subtracting max does not change probabilities.
pi_w = exp_w / exp_w.sum()
print("softmax probabilities:", np.round(pi_w, 3))
print("sum:", round(float(pi_w.sum()), 3))
assert np.allclose(np.round(pi_w, 3), [0.731, 0.269])

▶ What you'll see: action 0 receives about `73.1%` probability and action 1 receives about `26.9%`.

In [ ]:
action_rewards_w = np.array([2., 0.])
expected_reward_w = float(pi_w @ action_rewards_w)
print("expected immediate reward:", round(expected_reward_w, 3))
assert round(expected_reward_w, 3) == 1.462

▶ What you'll see: expected reward is `0.731·2 + 0.269·0 = 1.462`.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["a0", "a1"], pi_w, color="purple")
plt.title("4: policy probability mass")
plt.ylabel("π(a|s)")
plt.ylim(0, 1)
plt.show()

▶ What you'll see: most probability mass is on the higher-logit action, but the other action is still sampled.

*Why it's done this way:* action scores are easy to optimize, but decisions require probabilities if we want exploration and expectations. Softmax preserves ranking, normalizes to a distribution, and gives every action nonzero support unless a logit is driven infinitely far away.

### 5. Exploration pressure and policy support

A greedy agent can get stuck because it only samples what currently looks best. An exploration bonus deliberately rewards uncertainty. A UCB-style index is `mean + c * sqrt(2 log(t) / N)`, where rarely tried actions get a larger bonus. This does not say the action is good; it says information about the action is valuable.

In [ ]:
mean_w = 0.55
t_w = 20
count_w = 5
c_w = 1.0
bonus_w = c_w * np.sqrt(2 * np.log(t_w) / count_w)
ucb_w = mean_w + bonus_w
print("bonus:", round(bonus_w, 3), "UCB index:", round(ucb_w, 3))
assert round(ucb_w, 3) == 1.645

▶ What you'll see: the uncertainty bonus `1.095` makes the index much larger than the observed mean `0.55`.

In [ ]:
means_w = np.array([0.70, 0.55, 0.40])
counts_w = np.array([30, 5, 1])
indices_w = means_w + np.sqrt(2 * np.log(t_w) / counts_w)
print("UCB indices:", np.round(indices_w, 3))
print("chosen action:", int(np.argmax(indices_w)))

▶ What you'll see: the least-sampled action can win even with a lower mean because its uncertainty bonus is large.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["a0", "a1", "a2"], means_w, label="mean", color="gray")
plt.scatter(["a0", "a1", "a2"], indices_w, color="crimson", label="mean + bonus", zorder=3)
plt.title("5: uncertainty can change the selected action")
plt.ylabel("score")
plt.legend()
plt.show()

▶ What you'll see: red UCB dots sit above gray means, with the biggest lift on rarely sampled actions.

*Why it's done this way:* RL data depends on the policy. If an action is never sampled, its value is unsupported and cannot be learned from experience. Exploration bonuses temporarily pay for data, reducing the risk that early noise permanently hides a better long-term action.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, transition tables, probabilities, and simulations.
import matplotlib.pyplot as plt  # load Matplotlib for compact plots that debug RL quantities.
np.random.seed(0)  # make all random examples reproducible.

def discounted_return(rewards, gamma):  # compute G = sum_t gamma^t r_t for a finite reward list.
    rewards = np.asarray(rewards, dtype=float)  # ensure arithmetic uses floating point.
    powers = gamma ** np.arange(len(rewards))  # create discount weights 1, gamma, gamma^2, ... .
    return float(np.sum(powers * rewards))  # return the scalar discounted sum.

def softmax(logits):  # convert unconstrained action scores into probabilities.
    logits = np.asarray(logits, dtype=float)  # ensure vectorized floating-point operations.
    shifted = logits - np.max(logits)  # subtract the max for numerical stability without changing probabilities.
    exp_values = np.exp(shifted)  # exponentiate shifted scores into positive weights.
    return exp_values / exp_values.sum()  # normalize weights so they sum to 1.

def ucb_index(mean, count, t, c=1.0):  # compute an exploration-augmented action score.
    return mean + c * np.sqrt(2 * np.log(t) / count)  # add uncertainty bonus that shrinks with count.

def step_corridor(state, action):  # one deterministic toy environment step.
    transitions = np.array([[0, 1], [0, 2], [2, 2]])  # state-action next-state table.
    rewards = np.array([[0., 0.], [0., 1.], [0., 0.]])  # state-action reward table.
    next_state = int(transitions[state, action])  # read the next state.
    reward = float(rewards[state, action])  # read the reward.
    return next_state, reward  # return the environment response.

## 🟢 Basics (warm-up)

### Basic 1 — Build states and actions

**Goal.** Create the smallest RL vocabulary, because every later formula needs a state set, an action set, and tables whose shapes match those sets. We build it in 2 steps.

In [ ]:
states_b1 = np.array([0, 1, 2])  # define three states in a tiny corridor.
actions_b1 = np.array([0, 1])  # define two actions: left and right.
print("number of states:", len(states_b1))  # inspect |S|.
print("number of actions:", len(actions_b1))  # inspect |A|.

▶ What you'll see: the toy problem has `|S|=3` and `|A|=2`.

In [ ]:
Q_b1 = np.zeros((len(states_b1), len(actions_b1)))  # allocate one action value per state-action pair.
print("Q shape:", Q_b1.shape)  # inspect that Q has shape |S| x |A|.
assert Q_b1.shape == (3, 2)  # verify the state-action table shape.
plt.figure(figsize=(4, 3))
plt.imshow(Q_b1, cmap="viridis", aspect="auto")
plt.colorbar(label="Q value")
plt.title("Basic 1: empty state-action table")
plt.xlabel("action"); plt.ylabel("state")
plt.show()

▶ What you'll see: a 3×2 table initialized to zero.

👀 Takeaway: value tables must match the shape of the decision object they represent.

### Basic 2 — Encode one environment step

**Goal.** Store transition and reward tables, because an RL environment maps `(state, action)` into `(next_state, reward)`. We build it in 2 steps.

In [ ]:
T_b2 = np.array([[0, 1], [0, 2], [2, 2]])  # deterministic next-state table.
R_b2 = np.array([[0., 0.], [0., 1.], [0., 0.]])  # immediate reward table.
print("T[1,1] next state:", T_b2[1, 1])  # inspect moving right from the middle.
print("R[1,1] reward:", R_b2[1, 1])  # inspect the reward for that transition.

▶ What you'll see: moving right from state 1 reaches state 2 and pays reward 1.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(R_b2, cmap="YlGn", aspect="auto")
plt.colorbar(label="reward")
plt.title("Basic 2: reward table R[s,a]")
plt.xlabel("action"); plt.ylabel("state")
plt.show()
assert R_b2[1, 1] == 1.0

▶ What you'll see: only one bright cell has positive immediate reward.

👀 Takeaway: the environment, not the agent, defines the consequences of each action.

### Basic 3 — Roll out a fixed policy

**Goal.** Generate a short trajectory, because RL algorithms learn from sequences of experience rather than independent rows. We build it in 2 steps.

In [ ]:
s_b3 = 0  # start at the leftmost state.
traj_b3 = []  # store experience tuples.
for t_b3 in range(3):
    a_b3 = 1  # always choose right.
    s_next_b3, r_b3 = step_corridor(s_b3, a_b3)  # ask the environment for its response.
    traj_b3.append((s_b3, a_b3, r_b3, s_next_b3))  # record (s,a,r,s').
    s_b3 = s_next_b3  # advance to the next state.
print(traj_b3)

▶ What you'll see: the agent receives reward only after it has reached the middle state and moves right.

In [ ]:
rewards_b3 = np.array([row_b3[2] for row_b3 in traj_b3])  # extract rewards from the trajectory.
print("trajectory rewards:", rewards_b3)  # inspect the reward stream.
assert rewards_b3.sum() == 1.0  # verify the episode earned one unit of reward.
plt.figure(figsize=(4, 3))
plt.step(np.arange(len(rewards_b3)), rewards_b3, where="mid", color="teal")
plt.title("Basic 3: reward stream from a rollout")
plt.xlabel("time step"); plt.ylabel("reward")
plt.show()

▶ What you'll see: a delayed spike appears after the first positioning action.

👀 Takeaway: a trajectory shows why credit assignment is harder than one-step prediction.

### Basic 4 — Compute a finite discounted return

**Goal.** Turn a reward stream into a scalar return, because the agent optimizes cumulative consequence rather than one reward in isolation. We build it in 2 steps.

In [ ]:
rewards_b4 = np.array([1., 0., 2.])  # use the lesson reward stream.
gamma_b4 = 0.9  # choose the lesson discount.
powers_b4 = gamma_b4 ** np.arange(len(rewards_b4))  # compute [1, gamma, gamma^2].
print("powers:", np.round(powers_b4, 3))  # inspect discount multipliers.

▶ What you'll see: powers are `[1.000, 0.900, 0.810]`.

In [ ]:
G_b4 = discounted_return(rewards_b4, gamma_b4)  # compute sum gamma^t r_t.
print("return:", round(G_b4, 3))  # inspect the canonical return.
assert round(G_b4, 3) == 2.620  # verify the lesson number.
plt.figure(figsize=(4, 3))
plt.bar(["t0", "t1", "t2"], powers_b4 * rewards_b4, color="orange")
plt.title("Basic 4: discounted reward terms")
plt.ylabel("γ^t r_t")
plt.show()

▶ What you'll see: the reward two steps away contributes `1.62`.

👀 Takeaway: return is immediate reward plus discounted future reward.

### Basic 5 — Compare two discount factors

**Goal.** See how $\gamma$ changes planning horizon, because smaller discounts make delayed rewards matter less. We build it in 2 steps.

In [ ]:
rewards_b5 = np.array([0., 0., 10.])  # all payoff is delayed.
gammas_b5 = np.array([0.5, 0.9, 0.99])  # compare short to long horizon.
returns_b5 = np.array([discounted_return(rewards_b5, g_b5) for g_b5 in gammas_b5])  # compute each return.
print("returns:", np.round(returns_b5, 3))  # inspect delayed payoff under each gamma.

▶ What you'll see: the same delayed reward is worth `2.5`, `8.1`, or `9.801` depending on the horizon.

In [ ]:
assert np.allclose(np.round(returns_b5, 3), [2.5, 8.1, 9.801])  # verify concrete return values.
plt.figure(figsize=(4, 3))
plt.bar([str(g_b5) for g_b5 in gammas_b5], returns_b5, color="seagreen")
plt.title("Basic 5: discount controls horizon")
plt.xlabel("gamma"); plt.ylabel("return")
plt.show()

▶ What you'll see: larger discounts preserve more value for delayed payoff.

👀 Takeaway: γ is a modeling choice about how strongly the future should count.

### Basic 6 — Make a bootstrap target

**Goal.** Combine a one-step reward with a next-state estimate, because many RL methods learn before an episode finishes. We build it in 2 steps.

In [ ]:
r_b6 = 1.0  # observed immediate reward.
gamma_b6 = 0.9  # discount future value.
V_next_b6 = 0.8  # current estimate for the next state.
y_b6 = r_b6 + gamma_b6 * V_next_b6  # one-step target.
print("target y:", round(y_b6, 3))  # inspect r + gamma V(s').
assert round(y_b6, 3) == 1.720

▶ What you'll see: reward `1` plus discounted future estimate `0.72` gives target `1.72`.

In [ ]:
parts_b6 = np.array([r_b6, gamma_b6 * V_next_b6])  # split the target into immediate and future pieces.
plt.figure(figsize=(4, 3))
plt.bar(["reward", "discounted V(next)"], parts_b6, color=["gray", "teal"])
plt.title("Basic 6: one-step target pieces")
plt.ylabel("value")
plt.show()

▶ What you'll see: both immediate reward and estimated future value contribute to the target.

👀 Takeaway: bootstrapping learns from one real reward plus one learned estimate.

### Basic 7 — Apply a learning-rate update

**Goal.** Move an old estimate toward a target, because RL updates should be controlled rather than full overwrites. We build it in 2 steps.

In [ ]:
q_old_b7 = 0.4  # current action-value estimate.
target_b7 = 1.72  # one-step target from Basic 6.
alpha_b7 = 0.5  # learning rate.
td_error_b7 = target_b7 - q_old_b7  # signed gap from estimate to target.
print("TD error:", round(td_error_b7, 3))  # inspect the correction signal.

▶ What you'll see: the estimate is too low by `1.32`.

In [ ]:
q_new_b7 = q_old_b7 + alpha_b7 * td_error_b7  # partial move toward the target.
print("new Q:", round(q_new_b7, 3))  # inspect the updated estimate.
assert round(q_new_b7, 3) == 1.060  # verify the lesson number.
plt.figure(figsize=(4, 3))
plt.bar(["old", "target", "new"], [q_old_b7, target_b7, q_new_b7], color=["gray", "black", "green"])
plt.title("Basic 7: learning-rate controlled move")
plt.ylabel("Q value")
plt.show()

▶ What you'll see: the new estimate lands exactly halfway between old value and target.

👀 Takeaway: α controls how fast estimates react to new evidence.

### Basic 8 — Convert logits to a policy

**Goal.** Build a softmax policy, because RL agents often optimize scores but must sample actions from probabilities. We build it in 2 steps.

In [ ]:
logits_b8 = np.array([1., 0.])  # two action scores.
probs_b8 = softmax(logits_b8)  # convert scores into probabilities.
print("policy:", np.round(probs_b8, 3))  # inspect π(a|s).
assert np.allclose(np.round(probs_b8, 3), [0.731, 0.269])

▶ What you'll see: the higher logit receives more probability mass.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1"], probs_b8, color="purple")
plt.ylim(0, 1)
plt.title("Basic 8: softmax policy")
plt.ylabel("probability")
plt.show()

▶ What you'll see: probabilities sum to 1 but do not collapse to a hard argmax.

👀 Takeaway: a stochastic policy can prefer one action while still exploring another.

### Basic 9 — Compute expected reward under a policy

**Goal.** Weight rewards by action probabilities, because a stochastic policy's consequence is an expectation. We build it in 2 steps.

In [ ]:
probs_b9 = softmax(np.array([1., 0.]))  # reuse the lesson policy.
rewards_b9 = np.array([2., 0.])  # immediate rewards for the two actions.
contrib_b9 = probs_b9 * rewards_b9  # per-action expected reward contributions.
print("contributions:", np.round(contrib_b9, 3))  # inspect probability times reward.

▶ What you'll see: only action 0 contributes because action 1's reward is zero.

In [ ]:
expected_b9 = float(np.sum(contrib_b9))  # sum the weighted rewards.
print("expected reward:", round(expected_b9, 3))  # inspect the lesson number.
assert round(expected_b9, 3) == 1.462
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1"], contrib_b9, color="darkorange")
plt.title("Basic 9: expected reward terms")
plt.ylabel("π(a) r(a)")
plt.show()

▶ What you'll see: expected reward is the area represented by probability-weighted bars.

👀 Takeaway: policies affect learning by changing probability mass over consequences.

### Basic 10 — Add an exploration bonus

**Goal.** Compute a UCB score, because uncertainty can justify trying an action whose current mean is not best. We build it in 2 steps.

In [ ]:
mean_b10 = 0.55  # current empirical mean reward.
count_b10 = 5  # number of times the action was tried.
time_b10 = 20  # total decision count.
bonus_b10 = np.sqrt(2 * np.log(time_b10) / count_b10)  # uncertainty bonus.
print("bonus:", round(bonus_b10, 3))  # inspect exploration pressure.

▶ What you'll see: the bonus is bigger than one because the action has only five samples.

In [ ]:
ucb_b10 = mean_b10 + bonus_b10  # optimistic index.
print("UCB:", round(ucb_b10, 3))  # inspect the canonical UCB number.
assert round(ucb_b10, 3) == 1.645
plt.figure(figsize=(4, 3))
plt.bar(["mean", "bonus", "index"], [mean_b10, bonus_b10, ucb_b10], color=["gray", "teal", "crimson"])
plt.title("Basic 10: mean plus uncertainty")
plt.ylabel("score")
plt.show()

▶ What you'll see: the exploration index is the observed mean plus a temporary uncertainty lift.

👀 Takeaway: exploration bonuses turn missing information into a deliberate reason to act.

## 🟡 Easy

### Easy 1 — Evaluate a deterministic policy in the corridor

**Goal.** Compute returns for two fixed policies, because RL framing asks which action rule creates better long-run consequence. We build it in 3 steps.

In [ ]:
policies_e1 = {"always_left": np.array([0, 0, 0]), "always_right": np.array([1, 1, 1])}  # action per state.
gamma_e1 = 0.9  # discount for return calculation.
print("policies:", policies_e1)  # inspect deterministic action choices.

▶ What you'll see: one policy always chooses action 0, the other always chooses action 1.

In [ ]:
returns_e1 = []
for name_e1, policy_e1 in policies_e1.items():
    s_e1 = 0
    rewards_e1 = []
    for t_e1 in range(3):
        a_e1 = int(policy_e1[s_e1])
        s_e1, r_e1 = step_corridor(s_e1, a_e1)
        rewards_e1.append(r_e1)
    returns_e1.append(discounted_return(rewards_e1, gamma_e1))
print("returns:", dict(zip(policies_e1.keys(), np.round(returns_e1, 3))))
assert np.allclose(np.round(returns_e1, 3), [0.0, 0.9])

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(list(policies_e1.keys()), returns_e1, color=["gray", "green"])
plt.title("Easy 1: policy return comparison")
plt.ylabel("discounted return from start")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: always-right wins because it reaches the delayed reward.

👀 Takeaway: RL evaluates policies by consequence, not by whether the first action pays immediately.

### Easy 2 — Compute a Bellman expectation backup

**Goal.** Average over actions and next states, because the Bellman equation is just expected one-step reward plus discounted next value. We build it in 3 steps.

In [ ]:
pi_e2 = np.array([0.25, 0.75])  # in state 1, choose left with 25% and right with 75%.
V_e2 = np.array([0.3, 0.8, 0.0])  # current values for states 0, 1, 2.
gamma_e2 = 0.9  # discount future value.
print("policy at state 1:", pi_e2)  # inspect action weights.

▶ What you'll see: the policy mostly chooses right but still sometimes moves left.

In [ ]:
s_e2 = 1
T_e2 = np.array([[0, 1], [0, 2], [2, 2]])
R_e2 = np.array([[0., 0.], [0., 1.], [0., 0.]])
q_terms_e2 = np.array([R_e2[s_e2, a_e2] + gamma_e2 * V_e2[T_e2[s_e2, a_e2]] for a_e2 in [0, 1]])
backup_e2 = float(pi_e2 @ q_terms_e2)
print("action backup terms:", np.round(q_terms_e2, 3))
print("V backup:", round(backup_e2, 3))
assert round(backup_e2, 3) == 0.818

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["left term", "right term"], q_terms_e2, color="teal")
plt.title("Easy 2: Bellman action terms")
plt.ylabel("r + γV(s')")
plt.show()

▶ What you'll see: the right action has a larger backup term because it pays reward 1.

👀 Takeaway: Bellman expectation is a probability-weighted average over action consequences.

### Easy 3 — Update a full Q table from one transition

**Goal.** Modify exactly one state-action cell, because a Q update belongs to the action that was actually taken. We build it in 3 steps.

In [ ]:
Q_e3 = np.zeros((3, 2))  # initialize action values.
s_e3, a_e3, r_e3, sp_e3 = 1, 1, 1.0, 2  # observed transition from middle to goal.
gamma_e3 = 0.9
alpha_e3 = 0.5
print("before Q:\n", Q_e3)

▶ What you'll see: every state-action value starts at zero.

In [ ]:
target_e3 = r_e3 + gamma_e3 * np.max(Q_e3[sp_e3])  # Q-learning one-step target.
Q_e3[s_e3, a_e3] = Q_e3[s_e3, a_e3] + alpha_e3 * (target_e3 - Q_e3[s_e3, a_e3])
print("target:", round(target_e3, 3))
print("after Q:\n", Q_e3)
assert Q_e3[1, 1] == 0.5

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(Q_e3, cmap="viridis", aspect="auto")
plt.colorbar(label="Q value")
plt.title("Easy 3: one Q cell updated")
plt.xlabel("action"); plt.ylabel("state")
plt.show()

▶ What you'll see: only the `(state 1, right)` cell changes.

👀 Takeaway: Q tables store consequences for actions, so updates must preserve the state-action shape.

### Easy 4 — Simulate epsilon-greedy action choices

**Goal.** Mix greedy choice with random exploration, because a policy needs support for actions it has not fully learned yet. We build it in 3 steps.

In [ ]:
Q_e4 = np.array([[0.2, 0.8]])  # one state with two action values.
epsilon_e4 = 0.2  # probability of random exploration.
rng_e4 = np.random.default_rng(4)  # local reproducible generator.
print("Q values:", Q_e4[0], "epsilon:", epsilon_e4)

▶ What you'll see: action 1 is greedy, but exploration probability is nonzero.

In [ ]:
actions_e4 = []
for k_e4 in range(1000):
    if rng_e4.random() < epsilon_e4:
        actions_e4.append(int(rng_e4.integers(0, 2)))
    else:
        actions_e4.append(int(np.argmax(Q_e4[0])))
actions_e4 = np.array(actions_e4)
counts_e4 = np.bincount(actions_e4, minlength=2)
print("action counts:", counts_e4)
assert counts_e4[1] > counts_e4[0]

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1"], counts_e4 / counts_e4.sum(), color="purple")
plt.title("Easy 4: epsilon-greedy sampling")
plt.ylabel("empirical frequency")
plt.ylim(0, 1)
plt.show()

▶ What you'll see: action 1 dominates, while action 0 still receives samples.

👀 Takeaway: epsilon-greedy keeps policy support without ignoring current value estimates.

### Easy 5 — Compare immediate reward with return

**Goal.** Show why greedy immediate reward can lose, because a smaller first reward may unlock a larger later payoff. We build it in 3 steps.

In [ ]:
immediate_path_e5 = np.array([1., 0., 0.])  # take a quick small reward.
delayed_path_e5 = np.array([0., 0., 3.])  # wait for a larger payoff.
gamma_e5 = 0.9
print("first rewards:", immediate_path_e5[0], delayed_path_e5[0])

▶ What you'll see: the immediate path looks better if only the first reward is inspected.

In [ ]:
G_now_e5 = discounted_return(immediate_path_e5, gamma_e5)
G_later_e5 = discounted_return(delayed_path_e5, gamma_e5)
print("returns:", round(G_now_e5, 3), round(G_later_e5, 3))
assert round(G_later_e5, 3) == 2.430
assert G_later_e5 > G_now_e5

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["immediate", "delayed"], [G_now_e5, G_later_e5], color=["gray", "green"])
plt.title("Easy 5: return can reverse greedy reward")
plt.ylabel("discounted return")
plt.show()

▶ What you'll see: the delayed path wins after discounting even though its first reward is zero.

👀 Takeaway: RL framing optimizes return, so immediate reward is often the wrong target.

## 🔴 Advanced

### Advanced 1 — Estimate value by Monte Carlo rollouts

**Goal.** Average sampled returns, because value is an expectation over possible trajectories. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(11)  # reproducible rollout randomness.
gamma_a1 = 0.9
n_episodes_a1 = 500
print("episodes:", n_episodes_a1)

▶ What you'll see: the estimate will average many simulated returns from the start state.

In [ ]:
returns_a1 = []
for ep_a1 in range(n_episodes_a1):
    s_a1 = 0
    rewards_a1 = []
    for t_a1 in range(3):
        a_a1 = 1 if rng_a1.random() < 0.8 else 0  # mostly move right.
        s_a1, r_a1 = step_corridor(s_a1, a_a1)
        rewards_a1.append(r_a1)
    returns_a1.append(discounted_return(rewards_a1, gamma_a1))
returns_a1 = np.array(returns_a1)
print("mean return:", round(float(np.mean(returns_a1)), 3))
assert 0.45 < float(np.mean(returns_a1)) < 0.75

In [ ]:
running_a1 = np.cumsum(returns_a1) / np.arange(1, n_episodes_a1 + 1)
print("final running estimate:", round(float(running_a1[-1]), 3))

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(running_a1, color="teal")
plt.title("Advanced 1: Monte Carlo value estimate")
plt.xlabel("episode"); plt.ylabel("mean return")
plt.show()

▶ What you'll see: the running average is noisy at first and stabilizes as more episodes are observed.

👀 Takeaway: Monte Carlo values are unbiased samples of return but can have high variance.

### Advanced 2 — Sweep discount and observe policy preference

**Goal.** Change γ and watch which plan is preferred, because discounting can change the optimal action when rewards arrive at different times. We build it in 3 steps.

In [ ]:
gammas_a2 = np.linspace(0.1, 0.99, 30)  # horizon sweep.
short_a2 = np.array([1., 0., 0.])  # small reward now.
long_a2 = np.array([0., 0., 3.])  # larger reward later.
print("gamma range:", round(float(gammas_a2[0]), 2), "to", round(float(gammas_a2[-1]), 2))

▶ What you'll see: the sweep ranges from very short-sighted to long-horizon.

In [ ]:
short_returns_a2 = np.array([discounted_return(short_a2, g_a2) for g_a2 in gammas_a2])
long_returns_a2 = np.array([discounted_return(long_a2, g_a2) for g_a2 in gammas_a2])
prefer_long_a2 = long_returns_a2 > short_returns_a2
first_long_gamma_a2 = float(gammas_a2[np.argmax(prefer_long_a2)])
print("first sampled gamma preferring delayed plan:", round(first_long_gamma_a2, 3))
assert first_long_gamma_a2 > 0.55

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(gammas_a2, short_returns_a2, label="reward now")
plt.plot(gammas_a2, long_returns_a2, label="reward later")
plt.title("Advanced 2: discount changes preference")
plt.xlabel("gamma"); plt.ylabel("return")
plt.legend()
plt.show()

▶ What you'll see: the delayed plan overtakes the immediate plan only when γ is high enough.

👀 Takeaway: γ is not cosmetic; it can determine which behavior is rational under the objective.

### Advanced 3 — Compare UCB and greedy bandit choices

**Goal.** Simulate action selection with and without exploration bonuses, because uncertain actions need samples before their means are trusted. We build it in 4 steps.

In [ ]:
true_means_a3 = np.array([0.45, 0.55, 0.70])  # hidden reward probabilities.
rng_a3 = np.random.default_rng(13)
steps_a3 = 200
print("true best action:", int(np.argmax(true_means_a3)))

▶ What you'll see: action 2 is truly best, but the agent has to discover it.

In [ ]:
counts_ucb_a3 = np.ones(3)
means_ucb_a3 = np.array([0.0, 0.0, 0.0])
choices_ucb_a3 = []
for t_a3 in range(1, steps_a3 + 1):
    indices_a3 = means_ucb_a3 + np.sqrt(2 * np.log(t_a3 + 1) / counts_ucb_a3)
    a_a3 = int(np.argmax(indices_a3))
    reward_a3 = 1.0 if rng_a3.random() < true_means_a3[a_a3] else 0.0
    counts_ucb_a3[a_a3] += 1
    means_ucb_a3[a_a3] += (reward_a3 - means_ucb_a3[a_a3]) / counts_ucb_a3[a_a3]
    choices_ucb_a3.append(a_a3)
print("UCB counts:", counts_ucb_a3.astype(int))

In [ ]:
rng_greedy_a3 = np.random.default_rng(13)
counts_greedy_a3 = np.ones(3)
means_greedy_a3 = np.array([0.0, 0.0, 0.0])
choices_greedy_a3 = []
for t_greedy_a3 in range(1, steps_a3 + 1):
    a_greedy_a3 = int(np.argmax(means_greedy_a3))
    reward_greedy_a3 = 1.0 if rng_greedy_a3.random() < true_means_a3[a_greedy_a3] else 0.0
    counts_greedy_a3[a_greedy_a3] += 1
    means_greedy_a3[a_greedy_a3] += (reward_greedy_a3 - means_greedy_a3[a_greedy_a3]) / counts_greedy_a3[a_greedy_a3]
    choices_greedy_a3.append(a_greedy_a3)
print("greedy counts:", counts_greedy_a3.astype(int))
assert counts_ucb_a3[2] > counts_greedy_a3[2]

In [ ]:
plt.figure(figsize=(5, 3))
x_a3 = np.arange(3)
plt.bar(x_a3 - 0.18, counts_greedy_a3, width=0.36, label="greedy", color="gray")
plt.bar(x_a3 + 0.18, counts_ucb_a3, width=0.36, label="UCB", color="teal")
plt.xticks(x_a3, ["a0", "a1", "a2"])
plt.title("Advanced 3: exploration changes data coverage")
plt.ylabel("sample count")
plt.legend()
plt.show()

▶ What you'll see: UCB spreads samples more effectively and gives the best arm more chances.

👀 Takeaway: exploration is part of the data-generating process, not an afterthought.

### Advanced 4 — Visualize bootstrapping instability from a large step

**Goal.** Compare learning rates on the same target stream, because bootstrapping from moving estimates can amplify errors if steps are too aggressive. We build it in 4 steps.

In [ ]:
targets_a4 = np.array([1.72, 1.20, 1.60, 1.40, 1.50, 1.45])  # noisy one-step targets.
alphas_a4 = [0.2, 1.1]  # stable partial averaging versus overshooting.
print("targets:", targets_a4)

▶ What you'll see: targets fluctuate because each transition supplies a different bootstrap estimate.

In [ ]:
curves_a4 = []
for alpha_a4 in alphas_a4:
    q_a4 = 0.4
    curve_a4 = [q_a4]
    for target_a4 in targets_a4:
        q_a4 = q_a4 + alpha_a4 * (target_a4 - q_a4)
        curve_a4.append(q_a4)
    curves_a4.append(np.array(curve_a4))
print("final estimates:", [round(float(c_a4[-1]), 3) for c_a4 in curves_a4])

In [ ]:
assert abs(curves_a4[0][-1] - 1.18892352) < 1e-6
assert curves_a4[1].max() > targets_a4.max()

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(curves_a4[0], marker="o", label="α=0.2")
plt.plot(curves_a4[1], marker="o", label="α=1.1")
plt.title("Advanced 4: step size and bootstrap targets")
plt.xlabel("update number"); plt.ylabel("Q estimate")
plt.legend()
plt.show()

▶ What you'll see: the large step overshoots targets, while the smaller step smooths noisy updates.

👀 Takeaway: bootstrapped targets are useful but require conservative update control.

### Advanced 5 — Detect unsupported off-policy evaluation

**Goal.** Check policy support before trusting an estimate, because an action never sampled by the behavior policy cannot be evaluated from logged data alone. We build it in 4 steps.

In [ ]:
behavior_counts_a5 = np.array([95, 5, 0])  # logged action counts from data collection.
target_policy_a5 = np.array([0.2, 0.3, 0.5])  # policy we want to evaluate.
print("behavior counts:", behavior_counts_a5)
print("target policy:", target_policy_a5)

▶ What you'll see: the target policy wants action 2 half the time, but behavior data never sampled action 2.

In [ ]:
behavior_probs_a5 = behavior_counts_a5 / behavior_counts_a5.sum()
unsupported_a5 = (target_policy_a5 > 0) & (behavior_probs_a5 == 0)
print("behavior probabilities:", np.round(behavior_probs_a5, 3))
print("unsupported actions:", np.where(unsupported_a5)[0])
assert unsupported_a5[2]

In [ ]:
safe_target_a5 = target_policy_a5.copy()
safe_target_a5[unsupported_a5] = 0.0
safe_target_a5 = safe_target_a5 / safe_target_a5.sum()
print("renormalized supported target:", np.round(safe_target_a5, 3))
assert np.isclose(safe_target_a5.sum(), 1.0)

In [ ]:
plt.figure(figsize=(5, 3))
x_a5 = np.arange(3)
plt.bar(x_a5 - 0.18, behavior_probs_a5, width=0.36, label="behavior data", color="gray")
plt.bar(x_a5 + 0.18, target_policy_a5, width=0.36, label="target policy", color="crimson")
plt.xticks(x_a5, ["a0", "a1", "a2"])
plt.title("Advanced 5: policy support check")
plt.ylabel("probability")
plt.legend()
plt.show()

▶ What you'll see: action 2 has target probability but zero behavior support, making naive off-policy evaluation unsafe.

👀 Takeaway: logged RL estimates are only credible where the data-collection policy has support.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

The agent-environment loop turns delayed consequences into a discounted return.

RL framing belongs in Part 11 because reinforcement learning is where a prediction changes what data arrives next. Probability supplies transitions and expectations; optimization supplies iterative improvement for values, policies, and models. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

SEED = 1106
rng = np.random.default_rng(SEED)
GAMMA = 0.90
ACTIONS = ["up", "right", "down", "left"]
DELTAS = {
    "up": (-1, 0),
    "right": (0, 1),
    "down": (1, 0),
    "left": (0, -1),
}

@dataclass
class GridEnv:
    name: str
    rows: int
    cols: int
    start: int
    states: list
    index_of: dict
    terminal: set
    P: list
    rewards: np.ndarray
    shape_label: str


def discounted_return(rewards, gamma):
    total = 0.0
    power = 1.0
    for reward in rewards:
        total = total + power * reward
        power = power * gamma
    return total


def softmax(logits):
    shifted = np.asarray(logits, dtype=float) - np.max(logits)
    weights = np.exp(shifted)
    return weights / weights.sum()


def move_cell(cell, action, rows, cols, walls):
    dr, dc = DELTAS[action]
    nr = cell[0] + dr
    nc = cell[1] + dc
    candidate = (nr, nc)
    if nr < 0 or nr >= rows:
        return cell
    if nc < 0 or nc >= cols:
        return cell
    if candidate in walls:
        return cell
    return candidate


def build_grid_env(name, rows, cols, start_cell, goal_cells, pit_cells=None, walls=None, step_cost=-0.02, slip=0.0, wind=0.0, bonuses=None):
    pit_cells = set(pit_cells or [])
    walls = set(walls or [])
    bonuses = dict(bonuses or {})
    goal_cells = dict(goal_cells)
    states = []
    for r in range(rows):
        for c in range(cols):
            if (r, c) not in walls:
                states.append((r, c))
    index_of = {cell: i for i, cell in enumerate(states)}
    terminal_cells = set(goal_cells) | pit_cells
    terminal = {index_of[cell] for cell in terminal_cells}
    n_states = len(states)
    rewards = np.zeros(n_states)
    for cell, reward in goal_cells.items():
        rewards[index_of[cell]] = reward
    for cell in pit_cells:
        rewards[index_of[cell]] = -1.0
    for cell, reward in bonuses.items():
        rewards[index_of[cell]] = reward
    P = []
    for state_index, cell in enumerate(states):
        state_rows = []
        for action in ACTIONS:
            if state_index in terminal:
                state_rows.append([(1.0, state_index, 0.0, True)])
                continue
            side_actions = [action, ACTIONS[(ACTIONS.index(action) - 1) % 4], ACTIONS[(ACTIONS.index(action) + 1) % 4]]
            probs = [1.0 - slip, slip / 2.0, slip / 2.0]
            outcomes = {}
            for prob, actual_action in zip(probs, side_actions):
                if prob <= 0.0:
                    continue
                next_cell = move_cell(cell, actual_action, rows, cols, walls)
                windy_cell = move_cell(next_cell, "up", rows, cols, walls)
                wind_options = [(1.0 - wind, next_cell), (wind, windy_cell)]
                for wind_prob, final_cell in wind_options:
                    if wind_prob <= 0.0:
                        continue
                    next_index = index_of[final_cell]
                    done = next_index in terminal
                    reward = step_cost + rewards[next_index]
                    key = (next_index, done, reward)
                    outcomes[key] = outcomes.get(key, 0.0) + prob * wind_prob
            state_rows.append([(prob, ns, rew, done) for (ns, done, rew), prob in outcomes.items()])
        P.append(state_rows)
    shape_label = f"{rows}x{cols}, |S|={n_states}, |A|={len(ACTIONS)}"
    return GridEnv(name, rows, cols, index_of[start_cell], states, index_of, terminal, P, rewards, shape_label)


def two_state_chain():
    return build_grid_env(
        "D1 two-state chain",
        1,
        2,
        (0, 0),
        {(0, 1): 1.0},
        step_cost=0.0,
        slip=0.0,
    )


def build_env_ladder():
    envs = []
    envs.append(two_state_chain())
    envs.append(build_grid_env(
        "D2 slippery 3-state",
        1,
        3,
        (0, 0),
        {(0, 2): 1.0},
        pit_cells={(0, 1)},
        step_cost=-0.01,
        slip=0.20,
    ))
    envs.append(build_grid_env(
        "D3 4x4 gridworld",
        4,
        4,
        (3, 0),
        {(0, 3): 1.0},
        pit_cells={(1, 3)},
        walls={(1, 1), (2, 1)},
        step_cost=-0.03,
        slip=0.05,
    ))
    envs.append(build_grid_env(
        "D4 stochastic windy grid",
        5,
        5,
        (4, 0),
        {(0, 4): 1.2},
        pit_cells={(2, 3), (3, 2)},
        walls={(1, 1), (1, 2), (3, 1)},
        step_cost=-0.04,
        slip=0.15,
        wind=0.20,
    ))
    envs.append(build_grid_env(
        "D5 larger sparse-reward grid",
        8,
        8,
        (7, 0),
        {(0, 7): 2.0},
        pit_cells={(2, 5), (3, 5), (5, 3), (6, 6)},
        walls={(1, 1), (1, 2), (1, 3), (2, 1), (4, 2), (4, 3), (4, 4), (5, 5)},
        step_cost=-0.025,
        slip=0.10,
        wind=0.10,
        bonuses={(7, 1): 0.25},
    ))
    return envs


def q_from_v(env, V, gamma=GAMMA):
    Q = np.zeros((len(env.states), len(ACTIONS)))
    for s in range(len(env.states)):
        for a in range(len(ACTIONS)):
            total = 0.0
            for prob, next_state, reward, done in env.P[s][a]:
                total = total + prob * (reward + gamma * V[next_state] * (not done))
            Q[s, a] = total
    return Q


def policy_evaluation(env, policy, gamma=GAMMA, sweeps=200, tol=1e-10):
    V = np.zeros(len(env.states))
    errors = []
    for sweep in range(sweeps):
        old = V.copy()
        for s in range(len(env.states)):
            if s in env.terminal:
                V[s] = 0.0
                continue
            total = 0.0
            for a in range(len(ACTIONS)):
                for prob, next_state, reward, done in env.P[s][a]:
                    total = total + policy[s, a] * prob * (reward + gamma * old[next_state] * (not done))
            V[s] = total
        errors.append(float(np.max(np.abs(V - old))))
        if errors[-1] < tol:
            break
    return V, np.asarray(errors)


def value_iteration(env, gamma=GAMMA, sweeps=500, tol=1e-10):
    V = np.zeros(len(env.states))
    errors = []
    residuals = []
    for sweep in range(sweeps):
        old = V.copy()
        Q = q_from_v(env, old, gamma)
        for s in range(len(env.states)):
            if s in env.terminal:
                V[s] = 0.0
            else:
                V[s] = np.max(Q[s])
        residual = float(np.max(np.abs(V - old)))
        errors.append(residual)
        residuals.append(residual)
        if residual < tol:
            break
    policy = np.zeros((len(env.states), len(ACTIONS)))
    greedy = np.argmax(q_from_v(env, V, gamma), axis=1)
    for s, action in enumerate(greedy):
        policy[s, action] = 1.0
    return V, policy, np.asarray(errors), np.asarray(residuals)


def policy_iteration(env, gamma=GAMMA, sweeps=80):
    policy = np.ones((len(env.states), len(ACTIONS))) / len(ACTIONS)
    errors = []
    for sweep in range(sweeps):
        V, eval_errors = policy_evaluation(env, policy, gamma=gamma, sweeps=200)
        Q = q_from_v(env, V, gamma)
        greedy = np.argmax(Q, axis=1)
        new_policy = np.zeros_like(policy)
        for s, action in enumerate(greedy):
            new_policy[s, action] = 1.0
        change = float(np.max(np.abs(new_policy - policy)))
        errors.append(change)
        policy = new_policy
        if change == 0.0:
            break
    V, eval_errors = policy_evaluation(env, policy, gamma=gamma, sweeps=300)
    return V, policy, np.asarray(errors)


def run_episode(env, policy, gamma=GAMMA, max_steps=120, epsilon=0.0, start_state=None, rng=None):
    rng = rng or np.random.default_rng(SEED)
    state = env.start if start_state is None else start_state
    trajectory = []
    for step in range(max_steps):
        if state in env.terminal:
            break
        if rng.random() < epsilon:
            action = int(rng.integers(len(ACTIONS)))
        else:
            probs = policy[state] / policy[state].sum()
            action = int(rng.choice(len(ACTIONS), p=probs))
        choices = env.P[state][action]
        probs = np.array([item[0] for item in choices], dtype=float)
        probs = probs / probs.sum()
        choice = int(rng.choice(len(choices), p=probs))
        prob, next_state, reward, done = choices[choice]
        trajectory.append((state, action, reward, next_state, done))
        state = next_state
        if done:
            break
    return trajectory


def episode_return(trajectory, gamma=GAMMA):
    return discounted_return([step[2] for step in trajectory], gamma)


def monte_carlo_value(env, policy, episodes=300, gamma=GAMMA, epsilon=0.10, exploring_starts=False, rng=None):
    rng = rng or np.random.default_rng(SEED)
    V = np.zeros(len(env.states))
    counts = np.zeros(len(env.states))
    errors = []
    V_star, optimal_policy, vi_errors, residuals = value_iteration(env, gamma=gamma)
    for episode in range(episodes):
        start_state = None
        if exploring_starts:
            candidates = [s for s in range(len(env.states)) if s not in env.terminal]
            start_state = int(rng.choice(candidates))
        trajectory = run_episode(env, policy, gamma=gamma, epsilon=epsilon, start_state=start_state, rng=rng)
        G = 0.0
        seen = set()
        for state, action, reward, next_state, done in reversed(trajectory):
            G = reward + gamma * G
            if state in seen:
                continue
            seen.add(state)
            counts[state] = counts[state] + 1.0
            V[state] = V[state] + (G - V[state]) / counts[state]
        errors.append(float(np.max(np.abs(V - V_star))))
    return V, counts, np.asarray(errors)


def td0_value(env, policy, episodes=300, alpha=0.20, gamma=GAMMA, epsilon=0.10, rng=None):
    rng = rng or np.random.default_rng(SEED)
    V = np.zeros(len(env.states))
    errors = []
    V_star, optimal_policy, vi_errors, residuals = value_iteration(env, gamma=gamma)
    for episode in range(episodes):
        trajectory = run_episode(env, policy, gamma=gamma, epsilon=epsilon, rng=rng)
        for state, action, reward, next_state, done in trajectory:
            target = reward + gamma * V[next_state] * (not done)
            V[state] = V[state] + alpha * (target - V[state])
        errors.append(float(np.max(np.abs(V - V_star))))
    return V, np.asarray(errors)


def evaluate_policy_return(env, policy, episodes=80, gamma=GAMMA, rng=None):
    rng = rng or np.random.default_rng(SEED)
    returns = []
    for episode in range(episodes):
        trajectory = run_episode(env, policy, gamma=gamma, rng=rng)
        returns.append(episode_return(trajectory, gamma))
    return float(np.mean(returns))


def immediate_reward_policy(env):
    policy = np.zeros((len(env.states), len(ACTIONS)))
    for s in range(len(env.states)):
        if s in env.terminal:
            policy[s, 0] = 1.0
            continue
        means = []
        for a in range(len(ACTIONS)):
            means.append(sum(prob * reward for prob, next_state, reward, done in env.P[s][a]))
        policy[s, int(np.argmax(means))] = 1.0
    return policy


def uniform_policy(env):
    return np.ones((len(env.states), len(ACTIONS))) / len(ACTIONS)


def value_grid(env, V):
    grid = np.full((env.rows, env.cols), np.nan)
    for idx, cell in enumerate(env.states):
        grid[cell] = V[idx]
    return grid


def policy_grid(env, policy):
    chars = np.full((env.rows, env.cols), " ", dtype=object)
    arrows = np.array(["^", ">", "v", "<"], dtype=object)
    greedy = np.argmax(policy, axis=1)
    for idx, cell in enumerate(env.states):
        chars[cell] = "T" if idx in env.terminal else arrows[greedy[idx]]
    return chars


def plot_value_policy_panels(envs, values, policies, metric_values, metric_name):
    fig, axes = plt.subplots(2, len(envs), figsize=(4 * len(envs), 7))
    for i, env in enumerate(envs):
        ax = axes[0, i]
        image = ax.imshow(value_grid(env, values[i]), cmap="viridis")
        ax.set_title(env.name)
        plt.colorbar(image, ax=ax, fraction=0.046)
        ax = axes[1, i]
        ax.imshow(value_grid(env, values[i]), cmap="viridis")
        arrows = policy_grid(env, policies[i])
        grid = value_grid(env, values[i])
        for r in range(env.rows):
            for c in range(env.cols):
                if not np.isnan(grid[r, c]):
                    ax.text(c, r, arrows[r, c], ha="center", va="center", color="white")
        ax.set_title("greedy policy")
    fig.tight_layout()
    plt.figure(figsize=(7, 3))
    plt.plot(range(1, len(metric_values) + 1), metric_values, marker="o")
    plt.xticks(range(1, len(metric_values) + 1), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylabel(metric_name)
    plt.xlabel("environment rung")
    plt.title(f"{metric_name} across the D1-D5 ladder")
    plt.grid(True, alpha=0.3)
    plt.show()


def print_ladder_preview(envs):
    for env in envs:
        sample = env.states[: min(5, len(env.states))]
        print(f"{env.name}: {env.shape_label}; start={env.states[env.start]}; sample={sample}")


## The concept, built once on D1

We separate immediate reward from discounted return, then use the same environment ladder to show why delayed consequences matter.

Formula: $G=\sum_{t=0}^{T-1}\gamma^t r_{t+1}$ and $V^\pi(s)=\mathbb{E}_\pi[G_t\mid S_t=s]$

First assert the exact worked numbers from the lesson: discounted return, one-step TD target, softmax policy weighting, and UCB exploration pressure. These are small enough to verify by hand.

In [ ]:
lesson_return = discounted_return([1.0, 0.0, 2.0], 0.9)
td_target = 1.0 + 0.9 * 0.8
q_new = 0.4 + 0.5 * (td_target - 0.4)
policy_probs = softmax([1.0, 0.0])
expected_reward = policy_probs[0] * 2.0 + policy_probs[1] * 0.0
ucb_index = 0.55 + np.sqrt(2.0 * np.log(20.0) / 5.0)
assert np.isclose(lesson_return, 2.620)
assert np.isclose(td_target, 1.720)
assert np.isclose(q_new, 1.060)
assert np.isclose(np.round(policy_probs[0], 3), 0.731)
assert np.isclose(np.round(policy_probs[1], 3), 0.269)
assert np.isclose(np.round(expected_reward, 3), 1.462)
assert np.isclose(np.round(ucb_index, 3), 1.645)
print(lesson_return, td_target, q_new, policy_probs, expected_reward, ucb_index)

Start with the lesson's return formula. For rewards $[1,0,2]$ and $\gamma=0.9$, the exact return is $1+0.9\cdot0+0.9^2\cdot2=2.620$.

In [ ]:
def compute_return_and_value(rewards, gamma):
    G = discounted_return(rewards, gamma)
    immediate_reward = rewards[0]
    return immediate_reward, G

immediate_reward, G = compute_return_and_value([1.0, 0.0, 2.0], 0.9)
assert np.isclose(G, 2.620)
assert np.isclose(immediate_reward, 1.0)
print(f"immediate reward={immediate_reward:.3f}, discounted return={G:.3f}")

Now verify D1 by hand. In the 2-state chain, taking the right action reaches terminal reward $1$, so $V^*(s_0)=1$ and $V^*(s_1)=0$.

In [ ]:
env = two_state_chain()
V_star, policy_star, errors, residuals = value_iteration(env)
assert np.isclose(V_star[env.start], 1.0)
assert np.isclose(V_star[1], 0.0)
print(V_star)

## The dataset ladder

The family F12 ladder is built inline: D1 two-state chain, D2 slippery three-state, D3 4x4 gridworld, D4 stochastic windy grid, and D5 larger sparse-reward grid.

In [ ]:
envs = build_env_ladder()
print_ladder_preview(envs)

## Run the same method across D1-D5

Collect the plan metric: mean return.

In [ ]:
envs = build_env_ladder()
values = []
policies = []
metrics = []
for env in envs:
    V_star, policy_star, errors, residuals = value_iteration(env)
    mean_return = evaluate_policy_return(env, policy_star, rng=np.random.default_rng(SEED))
    values.append(V_star)
    policies.append(policy_star)
    metrics.append(mean_return)
    print(f"{env.name:28s}  {mean_return: .3f}")

## Results visualization

The closing figure has value/policy heatmap panels for every environment plus one summary curve over D1-D5.

In [ ]:
plot_value_policy_panels(envs, values, policies, metrics, "mean return")

## Pitfall on the hardest rung

Reproduce the named D5 pitfall, then apply the fix from the lesson.

In [ ]:
wrong_policy = immediate_reward_policy(envs[-1])
right_policy = policies[-1]
wrong_return = evaluate_policy_return(envs[-1], wrong_policy, rng=np.random.default_rng(SEED))
right_return = evaluate_policy_return(envs[-1], right_policy, rng=np.random.default_rng(SEED))
print(f"greedy immediate reward return: {wrong_return:.3f}")
print(f"discounted return policy return: {right_return:.3f}")
assert right_return > wrong_return

## Evaluate it + Practice

- Metric: mean return on D1-D5, compared with a no-skill uniform or immediate-reward baseline.
- Sanity check: D1 must match the hand value and the lesson numbers asserted above.
- Ablation: turn off discounted consequence or coverage and verify the metric worsens.
- Failure signal: residuals stop shrinking, value shapes mismatch, or D5 return drops below the baseline.
- Reproducibility: keep the provided seed and do not download simulators.

Practice prompts:
1. Change $\gamma$ from $0.90$ to $0.70$ and predict which rungs lose the most value before running.

2. Add one wall to D3 and inspect how the optimal policy heatmap reroutes around it.

3. On D5, compare the no-skill uniform policy with the learned or planned policy using the same return metric.